# 💬 YouTube Recent Videos Top Comments Extractor & Analytics

This Google Colab notebook retrieves the **top comments** (by relevance/likes) for the **10 most recent long-form videos** of a specified YouTube channel.

### 🌟 Key Features:
- **Google Drive Caching & Reuse**: Automatically checks Google Drive for existing video metadata or top comment datasets before executing YouTube API calls to avoid unnecessary quota usage.
- **API Quota Efficient**: Uses `channels.list`, `playlistItems.list`/`videos.list`, and `commentThreads.list` with `order="relevance"`.
- **Shorts Exclusion**: Leverages YouTube's long-form uploads playlist (`UULF` prefix) and duration checks (>60s) to strictly target recent long-form videos.
- **Google Drive Persistence**: Exports clean datasets in `.csv` and `.json` formats to `/content/drive/MyDrive/YouTube_Analytics/`.
- **Visual Comment Analytics**: Generates visual charts for top liked comments, comment distribution per video, character length metrics, and top author engagement.

## ⚙️ 1. Configuration & Setup

Set your YouTube API Key and Channel details below.
- **API Key**: You can store it in Colab Secrets (`YOUTUBE_API_KEY`) or paste it directly in `CONFIG["YOUTUBE_API_KEY"]` below.
- **Channel Identifier**: Can be a Channel Handle (e.g., `@josemillanastrologohumanista` or `@MKBHD`), Channel ID (e.g., `UC...`), or Username.

In [ ]:
# Install required packages if missing
!pip install -q google-api-python-client pandas matplotlib seaborn

import os
import re
import json
import datetime
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from googleapiclient.discovery import build
from googleapiclient.errors import HttpError

# Try loading API key from Colab Secrets if available
colab_api_key = ""
try:
    from google.colab import userdata
    colab_api_key = userdata.get('YOUTUBE_API_KEY')
except Exception:
    pass

# User Configuration
CONFIG = {
    "YOUTUBE_API_KEY": colab_api_key or "YOUR_YOUTUBE_API_KEY_HERE",
    "CHANNEL_IDENTIFIER": "@josemillanastrologohumanista",  # Channel Handle (@name), Channel ID (UC...), or Username
    "TARGET_VIDEO_COUNT": 10,                            # Number of recent long-form videos to collect comments for
    "MAX_COMMENTS_PER_VIDEO": 50,                        # Top comments to retrieve per video (ordered by relevance)
    "DRIVE_OUTPUT_DIR": "/content/drive/MyDrive/YouTube_Analytics",
    "MOUNT_DRIVE": True,                                 # Set to True when running in Google Colab
    "USE_CACHE": True                                    # Set to True to reuse cached datasets stored on Drive
}

print("Configuration loaded.")
if CONFIG["YOUTUBE_API_KEY"] in ["", "YOUR_YOUTUBE_API_KEY_HERE"]:
    print("⚠️ WARNING: Please set your YOUTUBE_API_KEY before running data fetching cells!")

## 💾 2. Mount Google Drive

Mount Google Drive to reuse cached datasets and persist output CSV and JSON files.

In [ ]:
if CONFIG["MOUNT_DRIVE"]:
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        os.makedirs(CONFIG["DRIVE_OUTPUT_DIR"], exist_ok=True)
        print(f"✅ Google Drive mounted successfully! Output directory: {CONFIG['DRIVE_OUTPUT_DIR']}")
    except Exception as e:
        print(f"ℹ️ Could not mount Google Drive automatically ({e}). Local directory will be used.")
        CONFIG["DRIVE_OUTPUT_DIR"] = "./output"
        os.makedirs(CONFIG["DRIVE_OUTPUT_DIR"], exist_ok=True)
else:
    os.makedirs(CONFIG["DRIVE_OUTPUT_DIR"], exist_ok=True)

## 🔍 3. YouTube API Client & Helper Functions

In [ ]:
def get_youtube_client(api_key):
    """Initializes YouTube Data API client."""
    return build('youtube', 'v3', developerKey=api_key)

def parse_iso8601_duration(duration_str):
    """Parses ISO 8601 duration string (e.g. PT1H2M3S) into total seconds."""
    pattern = re.compile(r'PT(?:(?P<hours>\d+)H)?(?:(?P<minutes>\d+)M)?(?:(?P<seconds>\d+)S)?')
    match = pattern.match(duration_str)
    if not match:
        return 0
    parts = match.groupdict()
    hours = int(parts['hours']) if parts['hours'] else 0
    minutes = int(parts['minutes']) if parts['minutes'] else 0
    seconds = int(parts['seconds']) if parts['seconds'] else 0
    return hours * 3600 + minutes * 60 + seconds

def format_seconds(seconds):
    """Formats duration in seconds to HH:MM:SS or MM:SS."""
    hours = seconds // 3600
    minutes = (seconds % 3600) // 60
    secs = seconds % 60
    if hours > 0:
        return f"{hours}:{minutes:02d}:{secs:02d}"
    return f"{minutes}:{secs:02d}"

def get_channel_info(youtube, identifier):
    """
    Resolves channel details using `channels.list` without search endpoints.
    Supports Handle (@name), Channel ID (UC...), or Username.
    """
    identifier = identifier.strip()
    res = None

    if identifier.startswith("@"):
        res = youtube.channels().list(part="snippet,contentDetails,statistics", forHandle=identifier).execute()
    elif identifier.startswith("UC"):
        res = youtube.channels().list(part="snippet,contentDetails,statistics", id=identifier).execute()
    else:
        res = youtube.channels().list(part="snippet,contentDetails,statistics", forUsername=identifier).execute()
        if not res.get("items"):
            res = youtube.channels().list(part="snippet,contentDetails,statistics", id=identifier).execute()

    if not res.get("items"):
        raise ValueError(f"Could not find channel for identifier: '{identifier}'. Check your input.")

    channel_item = res["items"][0]
    channel_id = channel_item["id"]
    uploads_playlist_id = channel_item["contentDetails"]["relatedPlaylists"]["uploads"]

    if uploads_playlist_id.startswith("UU"):
        longform_playlist_id = "UULF" + uploads_playlist_id[2:]
    elif channel_id.startswith("UC"):
        longform_playlist_id = "UULF" + channel_id[2:]
    else:
        longform_playlist_id = uploads_playlist_id

    return {
        "channel_id": channel_id,
        "title": channel_item["snippet"]["title"],
        "custom_url": channel_item["snippet"].get("customUrl", ""),
        "subscriber_count": channel_item["statistics"].get("subscriberCount", 0),
        "video_count": channel_item["statistics"].get("videoCount", 0),
        "uploads_playlist_id": uploads_playlist_id,
        "longform_playlist_id": longform_playlist_id
    }

## 📹 4. Retrieve Recent 10 Long-Form Videos (Drive Cache Aware)

To prevent redundant API queries, we first check Google Drive for existing video metadata exported by the analytics notebook. If missing or incomplete, we query the YouTube API.

In [ ]:
def load_cached_videos(output_dir, safe_channel_title, target_count=10):
    """Checks Google Drive for previously saved JSON video datasets."""
    if not os.path.exists(output_dir):
        return None
    for fname in os.listdir(output_dir):
        if fname.startswith(safe_channel_title) and fname.endswith(".json") and "comments" not in fname:
            fpath = os.path.join(output_dir, fname)
            try:
                with open(fpath, "r", encoding="utf-8") as f:
                    data = json.load(f)
                videos = data.get("videos", [])
                if len(videos) >= target_count:
                    print(f"⚡ Reusing cached video metadata from Google Drive: {fname}")
                    return videos[:target_count]
            except Exception as e:
                print(f"⚠️ Error reading cached file {fname}: {e}")
    return None

def fetch_last_n_longform_videos(youtube, playlist_id, fallback_playlist_id, target_count=10):
    """Fetches recent long-form (>60s) videos via YouTube Data API v3."""
    videos_data = []
    next_page_token = None
    current_playlist_id = playlist_id

    print(f"Fetching video items from playlist ID: {current_playlist_id}...")

    while len(videos_data) < target_count:
        try:
            pl_req = youtube.playlistItems().list(
                part="snippet,contentDetails",
                playlistId=current_playlist_id,
                maxResults=min(50, (target_count - len(videos_data)) * 3),
                pageToken=next_page_token
            )
            pl_res = pl_req.execute()
        except Exception as e:
            if current_playlist_id != fallback_playlist_id:
                print(f"⚠️ UULF playlist fetch issue ({e}). Falling back to main uploads playlist {fallback_playlist_id}...")
                current_playlist_id = fallback_playlist_id
                next_page_token = None
                continue
            else:
                raise e

        items = pl_res.get("items", [])
        if not items and len(videos_data) == 0 and current_playlist_id != fallback_playlist_id:
            print(f"⚠️ UULF playlist returned 0 items. Falling back to main uploads playlist {fallback_playlist_id}...")
            current_playlist_id = fallback_playlist_id
            next_page_token = None
            continue

        if not items:
            break

        video_ids = [item["contentDetails"]["videoId"] for item in items]

        v_req = youtube.videos().list(
            part="snippet,contentDetails,statistics",
            id=",".join(video_ids)
        )
        v_res = v_req.execute()

        for v_item in v_res.get("items", []):
            duration_iso = v_item["contentDetails"]["duration"]
            duration_sec = parse_iso8601_duration(duration_iso)

            if duration_sec <= 60:  # Exclude Shorts
                continue

            snippet = v_item["snippet"]
            stats = v_item.get("statistics", {})

            v_data = {
                "video_id": v_item["id"],
                "title": snippet.get("title"),
                "published_at": snippet.get("publishedAt"),
                "duration_iso": duration_iso,
                "duration_seconds": duration_sec,
                "duration_formatted": format_seconds(duration_sec),
                "view_count": int(stats.get("viewCount", 0)),
                "like_count": int(stats.get("likeCount", 0)),
                "comment_count": int(stats.get("commentCount", 0)),
                "thumbnail_url": snippet.get("thumbnails", {}).get("high", {}).get("url", ""),
                "url": f"https://www.youtube.com/watch?v={v_item['id']}"
            }
            videos_data.append(v_data)
            if len(videos_data) >= target_count:
                break

        next_page_token = pl_res.get("nextPageToken")
        if not next_page_token:
            break

    return videos_data[:target_count]

# Execute Channel & Video Retrieval
youtube = None
if CONFIG["YOUTUBE_API_KEY"] not in ["", "YOUR_YOUTUBE_API_KEY_HERE"]:
    youtube = get_youtube_client(CONFIG["YOUTUBE_API_KEY"])

channel_info = None
safe_channel_title = "channel"
if youtube:
    channel_info = get_channel_info(youtube, CONFIG["CHANNEL_IDENTIFIER"])
    safe_channel_title = re.sub(r'[^a-zA-Z0-9_-]', '_', channel_info['title'])
    print(f"Channel Identified: {channel_info['title']} (ID: {channel_info['channel_id']})")
else:
    print("ℹ️ No API key set yet. Attempting to locate cached data in Drive...")

recent_videos = None
if CONFIG["USE_CACHE"]:
    recent_videos = load_cached_videos(CONFIG["DRIVE_OUTPUT_DIR"], safe_channel_title, CONFIG["TARGET_VIDEO_COUNT"])

if not recent_videos:
    if not youtube:
        raise ValueError("API key required to fetch video data when no Drive cache is present.")
    print(f"Fetching recent {CONFIG['TARGET_VIDEO_COUNT']} long-form videos via API...")
    recent_videos = fetch_last_n_longform_videos(
        youtube,
        playlist_id=channel_info["longform_playlist_id"],
        fallback_playlist_id=channel_info["uploads_playlist_id"],
        target_count=CONFIG["TARGET_VIDEO_COUNT"]
    )

print(f"✅ Successfully selected {len(recent_videos)} recent long-form videos for comment extraction.")

## 💬 5. Extract Top Comments per Video (Drive Cache Aware)

We query `commentThreads.list` with `order="relevance"` to retrieve top-liked and most relevant top-level comments for each video. Drive cache is checked first to avoid unnecessary API requests.

In [ ]:
def load_cached_comments(output_dir, safe_channel_title):
    """Checks Google Drive for existing comments JSON dataset."""
    if not os.path.exists(output_dir):
        return None
    expected_fname = f"{safe_channel_title}_top_comments_10_recent_videos.json"
    fpath = os.path.join(output_dir, expected_fname)
    if os.path.exists(fpath):
        try:
            with open(fpath, "r", encoding="utf-8") as f:
                data = json.load(f)
            comments = data.get("comments", [])
            if comments:
                print(f"⚡ Reusing cached top comments dataset from Google Drive: {expected_fname}")
                return comments
        except Exception as e:
            print(f"⚠️ Error reading cached comments file {expected_fname}: {e}")
    return None

def fetch_top_comments_for_videos(youtube, videos, max_comments_per_video=50):
    """Fetches top comments (order=relevance) for a list of videos."""
    all_comments = []
    print(f"Fetching top comments for {len(videos)} videos (up to {max_comments_per_video} per video)...")

    for idx, video in enumerate(videos, start=1):
        v_id = video["video_id"]
        v_title = video.get("title", "")
        print(f" [{idx}/{len(videos)}] Fetching comments for: {v_title[:45]}... (ID: {v_id})")

        try:
            req = youtube.commentThreads().list(
                part="snippet",
                videoId=v_id,
                maxResults=min(50, max_comments_per_video),
                order="relevance"
            )
            res = req.execute()
            items = res.get("items", [])

            for item in items:
                top_snippet = item["snippet"]["topLevelComment"]["snippet"]
                comment_data = {
                    "video_id": v_id,
                    "video_title": v_title,
                    "video_publish_date": video.get("published_at", "")[:10],
                    "comment_id": item["id"],
                    "author_name": top_snippet.get("authorDisplayName", "Anonymous"),
                    "author_channel_url": top_snippet.get("authorChannelUrl", ""),
                    "author_profile_image": top_snippet.get("authorProfileImageUrl", ""),
                    "comment_text": top_snippet.get("textDisplay", ""),
                    "comment_text_clean": top_snippet.get("textOriginal", ""),
                    "like_count": int(top_snippet.get("likeCount", 0)),
                    "reply_count": int(item["snippet"].get("totalReplyCount", 0)),
                    "published_at": top_snippet.get("publishedAt"),
                    "updated_at": top_snippet.get("updatedAt")
                }
                all_comments.append(comment_data)

        except HttpError as e:
            if "commentsDisabled" in str(e):
                print(f"   ⚠️ Comments are disabled for video ID: {v_id}")
            else:
                print(f"   ⚠️ HTTP Error fetching comments for {v_id}: {e}")
        except Exception as e:
            print(f"   ⚠️ Unexpected error for {v_id}: {e}")

    return all_comments

# Execute Comment Fetching
comments_list = None
if CONFIG["USE_CACHE"]:
    comments_list = load_cached_comments(CONFIG["DRIVE_OUTPUT_DIR"], safe_channel_title)

if not comments_list:
    if not youtube:
        raise ValueError("API key required to fetch comments when no Drive cache is present.")
    comments_list = fetch_top_comments_for_videos(youtube, recent_videos, CONFIG["MAX_COMMENTS_PER_VIDEO"])

print(f"✅ Successfully retrieved {len(comments_list)} total top comments across recent videos.")

## 📊 6. Data Processing & Google Drive Export

In [ ]:
df_comments = pd.DataFrame(comments_list)

if not df_comments.empty:
    # Derived metrics
    df_comments["char_length"] = df_comments["comment_text_clean"].astype(str).str.len()
    df_comments["word_count"] = df_comments["comment_text_clean"].astype(str).str.split().str.len()
    df_comments["published_at_dt"] = pd.to_datetime(df_comments["published_at"])
    df_comments["comment_date"] = df_comments["published_at_dt"].dt.date

    # Define Export Paths
    csv_comments_path = os.path.join(CONFIG["DRIVE_OUTPUT_DIR"], f"{safe_channel_title}_top_comments_10_recent_videos.csv")
    json_comments_path = os.path.join(CONFIG["DRIVE_OUTPUT_DIR"], f"{safe_channel_title}_top_comments_10_recent_videos.json")

    # Export CSV
    df_comments.to_csv(csv_comments_path, index=False)

    # Export JSON
    export_json_comments = {
        "channel": channel_info or {"title": safe_channel_title},
        "extracted_at": datetime.datetime.now(datetime.timezone.utc).isoformat(),
        "video_count": len(recent_videos),
        "total_comments": len(df_comments),
        "videos": recent_videos,
        "comments": comments_list
    }
    with open(json_comments_path, "w", encoding="utf-8") as f:
        json.dump(export_json_comments, f, indent=2, ensure_ascii=False)

    print(f"📁 Datasets successfully saved to Google Drive:")
    print(f"   - CSV:  {csv_comments_path}")
    print(f"   - JSON: {json_comments_path}")

    # Display Preview of top liked comments
    print("\n🔥 Top 5 Most Liked Comments Preview:")
    preview_cols = ["author_name", "like_count", "reply_count", "video_title", "comment_text_clean"]
    display(df_comments.sort_values(by="like_count", ascending=False)[preview_cols].head(5))
else:
    print("ℹ️ No comments available to export.")

## 📈 7. Visual Comment Analytics

Visualizations analyzing top comment likes, comment counts per video, character length distribution, and commenter activity.

In [ ]:
if not df_comments.empty:
    sns.set_theme(style="whitegrid", palette="muted")
    plt.rcParams.update({'font.size': 10, 'figure.titlesize': 14})

    # Chart 1: Top 10 Most Liked Comments Across Recent Videos
    plt.figure(figsize=(12, 6))
    top_liked = df_comments.sort_values(by="like_count", ascending=False).head(10).copy()
    top_liked["label"] = top_liked["author_name"] + " (" + top_liked["video_title"].str[:25] + "...)"
    ax = sns.barplot(x="like_count", y="label", data=top_liked, palette="Purples_r")
    plt.title("Top 10 Most Liked Comments Across Recent 10 Videos", fontsize=14, fontweight="bold", pad=15)
    plt.xlabel("Comment Like Count", fontweight="bold")
    plt.ylabel("")
    for p in ax.patches:
        width = p.get_width()
        ax.annotate(f"{int(width):,}",
                    (width, p.get_y() + p.get_height() / 2.),
                    ha='left', va='center',
                    xytext=(5, 0),
                    textcoords='offset points',
                    fontsize=9)
    plt.tight_layout()
    plt.show()

    # Chart 2: Extracted Comments per Video
    plt.figure(figsize=(12, 5))
    comment_counts = df_comments["video_title"].value_counts().reset_index()
    comment_counts.columns = ["video_title", "count"]
    ax = sns.barplot(x="count", y="video_title", data=comment_counts, palette="Teal_r")
    plt.title("Extracted Top Comments Count per Video", fontsize=14, fontweight="bold", pad=15)
    plt.xlabel("Top Comments Collected", fontweight="bold")
    plt.ylabel("")
    for p in ax.patches:
        width = p.get_width()
        ax.annotate(f"{int(width)}",
                    (width, p.get_y() + p.get_height() / 2.),
                    ha='left', va='center',
                    xytext=(5, 0),
                    textcoords='offset points',
                    fontsize=9)
    plt.tight_layout()
    plt.show()

    # Chart 3: Comment Character Length Distribution
    plt.figure(figsize=(10, 5))
    sns.histplot(df_comments["char_length"], bins=30, kde=True, color="#2b5c8f")
    plt.title("Distribution of Comment Character Lengths", fontsize=14, fontweight="bold", pad=15)
    plt.xlabel("Comment Length (Characters)", fontweight="bold")
    plt.ylabel("Frequency", fontweight="bold")
    plt.tight_layout()
    plt.show()

    # Chart 4: Comment Likes vs Replies Scatter Plot
    plt.figure(figsize=(10, 5))
    sns.scatterplot(x="like_count", y="reply_count", size="char_length", sizes=(40, 300), hue="char_length", data=df_comments, palette="viridis")
    plt.title("Comment Likes vs. Reply Count", fontsize=14, fontweight="bold", pad=15)
    plt.xlabel("Comment Like Count", fontweight="bold")
    plt.ylabel("Reply Count", fontweight="bold")
    plt.legend(title="Char Length", bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.tight_layout()
    plt.show()